# Week 11 - Visualization with scanpy, part 2

After installing Scanpy and matplotlib to your fresh environment from anaconda prompt, import both modules:

In [ ]:

from __future__ import annotations

import scanpy as sc
from matplotlib.pyplot import rc_context

## define parameters for plotting
sc.set_figure_params(dpi=100, color_map="viridis_r")
sc.settings.verbosity = 0
sc.logging.print_header()


# Load the tutorial dataset

We will be using the human PBMC dataset. This dataset has been preprocessed and reduced in size for convenience


In [ ]:
pbmc = sc.datasets.pbmc68k_reduced()

# inspect the layers of the AnnData object. What layer are your feature/gene counts stored in?
pbmc

An important base function in sc transcriptomic analysis is to display the expression of a feature/gene by cluster in UMAP space

The shell code for this function looks like this:


In [ ]:
# rc_context is used for the figure size, in this case 4x4
with rc_context({"figure.figsize": (4, 4)}):
    sc.pl.umap(pbmc, color="CD79A")


## Exercise

The above code simply assigns the 'color' variable to the fold expression of our marker gene for the CD79A annotated cluster.
As we expect, it is expressed highly in the CD79A cluster, and at relatively low levels elsewhere

what if you were to output a scatterplot showing the expression of a non-marker gene IL6  in UMAP space?


In [ ]:
# Answer

# again, rc_context is used for the figure size, in this case 4x4
with rc_context({"figure.figsize": (4, 4)}):
    sc.pl.umap(pbmc, color="IL7R")


The scatterplot function is versitile, and not limited to plotting expression or markers in UMAP space
For instance, we can generate a plot with our color variable assigned to celltype annotations from this dataset's metadata,
multiple markers, and gene counts (a fairly silly metric to plot in this way).


In [ ]:
color_vars = [
    "CD79A",
    "MS4A1",
    "IGJ",
    "CD3D",
    "FCER1A",
    "FCGR3A",
    "n_counts",
    "bulk_labels",
]
with rc_context({"figure.figsize": (3, 3)}):
    sc.pl.umap(pbmc, color=color_vars, s=50, frameon=False, ncols=4, vmax="p99")


Before we go futher, let's take our cluster markers and make some proper celltype annotations for our data:


In [ ]:
marker_genes_dict = {
    "B-cell": ["CD79A", "MS4A1"],
    "Dendritic": ["FCER1A", "CST3"],
    "Monocytes": ["FCGR3A"],
    "NK": ["GNLY", "NKG7"],
    "Other": ["IGLL1"],
    "Plasma": ["IGJ"],
    "T-cell": ["CD3D"],
}
## Here we are just refining our annotations based on the exclusivity of marker expression
# create a dictionary to map cluster to annotation label
cluster2annotation = {
    "0": "Monocytes",
    "1": "NK",
    "2": "T-cell",
    "3": "Dendritic",
    "4": "Dendritic",
    "5": "Plasma",
    "6": "B-cell",
    "7": "Dendritic",
    "8": "Other",
}


Now define our clusters, and save to "observations" AnnData layer:


In [ ]:
# compute clusters using the leiden method and store the results with the name `clusters`
sc.tl.leiden(
    pbmc,
    key_added="clusters",
    resolution=0.5,
    n_iterations=2,
    flavor="igraph",
    directed=False,
)


Add a new `.obs` column called `cell type` by mapping clusters to annotation using pandas `map` function


In [ ]:
pbmc.obs["cell type"] = pbmc.obs["clusters"].map(cluster2annotation).astype("category")


# Dot plots

Now let's move on to some data visualization strategies to help us compare population expression magnitude between multiple clusters

One approach to this is the dotplot


In [ ]:
## The basic format for generating a dotplot in Scanpy is:
sc.pl.dotplot(pbmc, marker_genes_dict, "clusters", dendrogram=True)


Now let's look at the marker dotplot again:

In [ ]:

sc.pl.dotplot(pbmc, marker_genes_dict, "cell type", dendrogram=True)


What if we want to compare expression of a gene of interest accross these clusters?

First, define our genes to plot:

In [ ]:

goi_interleukin = ['IL1B', 'PILRA', 'IL2RG', 'IL18', 'IL16', 'IL32']

sc.pl.dotplot(pbmc, goi_interleukin, "cell type", dendrogram=True)


## Exercise 
How about your favorite gene of interest? How would you create a dotplot to see its cluster-level expression in immune cells?


In [ ]:
# print available genes
print(list(pbmc.var_names))

In [ ]:
# Answer

mynewgoi = ['insert fav genes here']
sc.pl.dotplot(pbmc, mynewgoi, "cell type", dendrogram=True)

# Heatmaps

Dotplots are fantastic to look at an overview of transcriptome patterns at the population level, but what if you want a 
survey of what expression looks like for your gene of interest from cell to cell?

Here we can employ a heatmap, which outputs a dense matrix plot showing the fold expression (or log fold change if comparing two conditions!)
of a gene of interest in each cluster.

The basic format for making a heatmap in Scanpy is:


In [ ]:

sc.pl.heatmap(pbmc, marker_genes_dict,"cell type" , swap_axes=True)


## Exercise 

Notice this is just plotting our cluster markers, but it still gives a sense of the interclustal heterogeneity,
even in features that we used to define our clusters identity.
also, we flipped it 90 degrees using swap_axes to make it less horrid to read. If your Y-axis variable (for heatmaps, the clusters you wish to interrogate)
is shorter, there is no need to do this

What if I wanted to make a heatmap comparing the transcriptional heterogeneity of our interleukin-related genes of interest from earlier?


In [ ]:
# Answer
sc.pl.heatmap(pbmc, goi_interleukin,"cell type" , swap_axes=True)


Curious about your favorite gene of interest in a particular cluster? Try making a new dictionary for your gene and running the same code to make a plot


# More resources

Great job mastering the basics of plotting SC transcriptome data with scanpy.
Follow this link to see a tutorial for a (more time consuming) vignette I'd like to touch on quickly-

RNA velocity modeling: https://scanpy.readthedocs.io/en/stable/tutorials/trajectories/paga-paul15.html